In [16]:
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
from collections import deque

env = gym.make("CarRacing-v3", render_mode="human")
    
""" 
Observation Space: 96x96 RGB Image
Action Space: [float32: steering direction(-1 = left to 1 = right), gas(0 to 1), braking(0 to 1)]

"""

def test_run(n_ep, env):
    for _ in range(n_ep):
        observation, info = env.reset()
        rewards = []
        hi = 0
        while True:
            hi += 1
            # action = env.action_space.sample()
            action = np.array([(np.random.rand() * 2 - 1) * 0.05, 1, 0.])
            observation, reward, terminated, truncated, info = env.step(action) 
            done = terminated or truncated
            rewards.append(reward) 

            if done or hi > 100:
                break
        return rewards

rewards = test_run(1, env)
env.close()

# cum_rewards = [np.sum(rewards[:i]) for i in range(len(rewards))]

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

# ax1.plot(rewards)
# ax2.plot(cum_rewards);

In [17]:
env = gym.make("CarRacing-v3")

def generate_episode(n_ep, env):
    for _ in range(n_ep):
        observation, info = env.reset()
        observations = []
        while True:
            # action = env.action_space.sample()
            action = np.array([(np.random.rand() * 2 - 1) * 0.05, 1, 0.])
            observation, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            observations.append(observation)
            if done:
                break
    return observations

observations = generate_episode(1, env)
observations = torch.tensor(np.array(observations))

In [18]:
class Encoder(nn.Module):
    def __init__(self, img_channels=3, n_hidden=32):
        super().__init__()
        self.n_hidden = n_hidden
        self.img_channels = img_channels
        
        self.conv = nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=4, stride=2), 
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),           
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2),          
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2),       
            nn.ReLU()
        )
        
        self.mu = nn.Linear(4096, n_hidden)
        self.logstd = nn.Linear(4096, n_hidden)
        
    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        
        x = self.conv(x)
        x = x.reshape(x.size(0), -1)

        mu = self.mu(x)
        logstd = self.logstd(x)
        return mu, logstd

class Decoder(nn.Module):
    def __init__(self, img_channels=3, n_hidden=32):
        super().__init__()
        self.img_channels = img_channels
        self.fc = nn.Linear(n_hidden, 4096)
        
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2), 
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2),   
            nn.ReLU(),
            nn.ConvTranspose2d(32, img_channels, kernel_size=6, stride=2), 
        )
            
    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.shape[0], 256, 4, 4)
        x = self.deconv(x)

        x = x.permute(0, 2, 3, 1)
        return x

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.Encoder = Encoder(3, n_hidden = 32)
        self.Decoder = Decoder(3, n_hidden = 32)

    def forward(self, x):
        mu, logstd = self.Encoder(x)
        
        std = logstd.exp()
        epsilon = torch.randn_like(std)
        z = mu + epsilon * std

        decoded = self.Decoder(z)
        return mu, logstd, decoded

In [24]:
from torch.utils.data import DataLoader

def loss_fn(data, reconst, mu, logstd):
    # reconst_loss = reconst_loss = nn.functional.binary_cross_entropy_with_logits(reconst, data, reduction='sum')
    bce = torch.nn.functional.mse_loss(reconst, data, reduction='sum')
    KL_Divergence = -0.5 * torch.sum(1 + 2 * logstd - mu ** 2 - torch.exp(2 * logstd))

    return bce + KL_Divergence

dataset = observations
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = VAE()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_ep = 50

for epoch in range(n_ep):
    total_loss = 0.0
    for batch_idx, data in enumerate(loader):
        # print(data.shape)
        # data_i = data[0].view(data[0].size(0), -1)
        optimizer.zero_grad()
        
        mu, logstd, decoded = model(data.float() / 256)
        loss = loss_fn(data.float() / 256, decoded, mu, logstd)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(loader.dataset)
    if epoch % 5 == 0:
        print(
            "Epoch {}/{}: loss={:.4f}".format(epoch + 1, n_ep, epoch_loss)
        )

Epoch 1/50: loss=3511.2853
Epoch 6/50: loss=199.5985
Epoch 11/50: loss=100.1813
Epoch 16/50: loss=72.4600
Epoch 21/50: loss=58.9460
Epoch 26/50: loss=51.8827
Epoch 31/50: loss=51.4526
Epoch 36/50: loss=45.2284
Epoch 41/50: loss=44.1626
Epoch 46/50: loss=39.4185


In [31]:
img = observations[7000].float() / 256
mu, logstd, decoded = model(img.view(1, 96, 96, 3))

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(img)
ax2.imshow(decoded.view(96, 96, 3).detach().cpu() , cmap='viridis');

IndexError: index 7000 is out of bounds for dimension 0 with size 1000

In [18]:
torch.save(model.state_dict(), "models/CarRacing_VAE_30_epochs")

In [2]:
env = gym.make("CarRacing-v3", render_mode="human")

def generate_episode(n_ep, env):
    for _ in range(n_ep):
        observation, info = env.reset()
        observations = []
        while True:
            action = env.action_space.sample()
            observation, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            observations.append(observation)
            if done:
                break
    return observations

generate_episode(1, env)
env.close()